In [1]:
import numpy as np

# ============================================================================
# TEXTURE 6: Oval strips (stretched in x direction) and cloud point patterns
# ============================================================================
# Same dimensions as previous textures
num_channels = 4
value_range = (0, 10)
z_range = (0, 48)
y_range = (0, 343)
x_range = (0, 680)

num_values = value_range[1] - value_range[0] + 1
z_dim = z_range[1] - z_range[0] + 1
y_dim = y_range[1] - y_range[0] + 1
x_dim = x_range[1] - x_range[0] + 1

# Create 5D array initialized with zeros
data = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)

# Common center for all channels
base_center_x = x_dim // 2
base_center_y = y_dim // 2

# Parameters for strips and cloud points
radius_ch0_inner = 50  # Channel 0: inner radius
radius_ch0_outer = 100  # Channel 0: outer radius (strip width = 20)
radius_ch1_inner = 80  # Channel 1: inner radius
radius_ch1_outer = 110  # Channel 1: outer radius (strip width = 20)
cloud_radius = 20  # Radius for cloud points (concentrated)
target_radius = 80  # Target radius for cloud points (channel 2 and 3)
num_cloud_points = 5  # Number of cloud point centers per channel
points_per_cloud = 100  # Points per cloud (concentrated, low scattering)

def create_concentrated_cloud_point(center_x, center_y, center_z, radius, num_points):
    """Create a concentrated cloud of points around a center (low scattering)"""
    cloud_points = []
    for _ in range(num_points):
        # Use square root distribution for more concentration near center
        angle = np.random.uniform(0, 2 * np.pi)
        r = radius * np.sqrt(np.random.uniform(0, 1))  # Concentrated distribution
        x = int(center_x + r * np.cos(angle))
        y = int(center_y + r * np.sin(angle))
        z = center_z
        if 0 <= x < x_dim and 0 <= y < y_dim:
            cloud_points.append((x, y, z))
    return cloud_points

def create_concentrated_3d_cloud_point(center_x, center_y, center_z, radius_xy, radius_z, num_points, z_dim):
    """Create a concentrated 3D cloud of points around a center (low scattering in 3D)"""
    cloud_points = []
    for _ in range(num_points):
        # Use square root distribution for more concentration near center in XY plane
        angle = np.random.uniform(0, 2 * np.pi)
        r_xy = radius_xy * np.sqrt(np.random.uniform(0, 1))  # Concentrated distribution in XY
        x = int(center_x + r_xy * np.cos(angle))
        y = int(center_y + r_xy * np.sin(angle))
        
        # Z distribution: uniform in [-radius_z, radius_z] for 3D cloud
        dz = int(np.random.uniform(-radius_z, radius_z))
        z = center_z + dz
        z = int(np.clip(z, 0, z_dim - 1))  # Clip to valid z range
        
        if 0 <= x < x_dim and 0 <= y < y_dim and 0 <= z < z_dim:
            cloud_points.append((x, y, z))
    return cloud_points

# Generate texture6: Oval strips (stretched in x direction) and cloud patterns
x_stretch = 1.5  # Stretch factor in x direction (x_radius = y_radius * x_stretch for oval)

for z in range(z_dim):
    # Channel 0: Oval strip (stretched in x direction) from r_inner to r_outer
    # Use elliptical distance: (x/a)^2 + (y/b)^2 where a = x_stretch * radius, b = radius
    for y in range(y_dim):
        for x in range(x_dim):
            dx = (x - base_center_x) / x_stretch  # Scale x by stretch factor
            dy = y - base_center_y
            # Elliptical distance (normalized)
            elliptical_dist = np.sqrt(dx**2 + dy**2)
            if radius_ch0_inner <= elliptical_dist <= radius_ch0_outer:
                random_value = np.random.randint(6, 11)
                data[0, random_value, z, y, x] = 1
    
    # Channel 1: Oval strip (stretched in x direction) from r_inner to r_outer
    for y in range(y_dim):
        for x in range(x_dim):
            dx = (x - base_center_x) / x_stretch  # Scale x by stretch factor
            dy = y - base_center_y
            # Elliptical distance (normalized)
            elliptical_dist = np.sqrt(dx**2 + dy**2)
            if radius_ch1_inner <= elliptical_dist <= radius_ch1_outer:
                random_value = np.random.randint(6, 11)
                data[1, random_value, z, y, x] = 1
    
# Channel 2 and 3: 3D concentrated cloud points centered at z = z_max/2, distributed in ring between radius_ch1_inner and radius_ch1_outer
z_center = z_dim // 2  # Middle z slice (z = z_max/2)
z_spread = 10  # Spread in z direction: ±10 from z_center (for radius 20)
max_centroid_distance = 5  # Maximum distance between channel 2 and 3 centroids

# Store channel 2 centroids first
ch2_centroids = []

# Generate 3D clouds for Channel 2 (distributed in oval ring)
for _ in range(num_cloud_points):
    # Random angle and radius in the ring between radius_ch1_inner and radius_ch1_outer
    cloud_angle = np.random.uniform(0, 2 * np.pi)
    # Uniform distribution in ring area (radius between inner and outer)
    cloud_radial_distance = np.random.uniform(radius_ch1_inner+5, radius_ch1_inner-5)
    
    # Use oval distribution: stretch x direction
    cloud_center_x = int(base_center_x + cloud_radial_distance * x_stretch * np.cos(cloud_angle))
    cloud_center_y = int(base_center_y + cloud_radial_distance * np.sin(cloud_angle))
    cloud_center_x = np.clip(cloud_center_x, cloud_radius, x_dim - cloud_radius - 1)
    cloud_center_y = np.clip(cloud_center_y, cloud_radius, y_dim - cloud_radius - 1)
    
    # Store channel 2 centroid
    ch2_centroids.append((cloud_center_x, cloud_center_y))
    
    # Create 3D concentrated cloud around this center for Channel 2
    # cloud_radius is for XY plane, z_spread is for Z direction
    cloud_points = create_concentrated_3d_cloud_point(
        cloud_center_x, cloud_center_y, z_center, cloud_radius, z_spread, points_per_cloud, z_dim
    )
    for x, y, z_val in cloud_points:
        if 0 <= x < x_dim and 0 <= y < y_dim and 0 <= z_val < z_dim:
            random_value = np.random.randint(6, 11)
            data[2, random_value, z_val, y, x] = 1

# Generate 3D clouds for Channel 3 - near channel 2 centroids (max distance 10)
for ch2_x, ch2_y in ch2_centroids:
    # Create channel 3 centroid near channel 2 centroid (within max_centroid_distance)
    # Random offset in XY plane, maximum distance 10
    offset_angle = np.random.uniform(0, 2 * np.pi)
    offset_distance = np.random.uniform(0, max_centroid_distance)
    
    ch3_center_x = int(ch2_x + offset_distance * np.cos(offset_angle))
    ch3_center_y = int(ch2_y + offset_distance * np.sin(offset_angle))
    ch3_center_x = np.clip(ch3_center_x, cloud_radius, x_dim - cloud_radius - 1)
    ch3_center_y = np.clip(ch3_center_y, cloud_radius, y_dim - cloud_radius - 1)
    
    # Create 3D concentrated cloud around this center for Channel 3
    # cloud_radius is for XY plane, z_spread is for Z direction
    cloud_points = create_concentrated_3d_cloud_point(
        ch3_center_x, ch3_center_y, z_center, cloud_radius, z_spread, points_per_cloud, z_dim
    )
    for x, y, z_val in cloud_points:
        if 0 <= x < x_dim and 0 <= y < y_dim and 0 <= z_val < z_dim:
            random_value = np.random.randint(6, 11)
            data[3, random_value, z_val, y, x] = 1

print(f"Created texture6 (Oval strips and cloud patterns): {data.shape}")
print(f"Dimensions breakdown:")
print(f"  - Channels: {num_channels} (1, 2, 3, 4)")
print(f"  - Values: {num_values} (0 to {value_range[1]})")
print(f"  - Z: {z_dim} (0 to {z_range[1]})")
print(f"  - Y: {y_dim} (0 to {y_range[1]})")
print(f"  - X: {x_dim} (0 to {x_range[1]})")
print(f"\nTexture6 parameters:")
print(f"  - Center position: ({base_center_x}, {base_center_y})")
print(f"  - Channel 0: Oval strip (stretched in x by {x_stretch}x) from r={radius_ch0_inner} to r={radius_ch0_outer} (width={radius_ch0_outer-radius_ch0_inner})")
print(f"  - Channel 1: Oval strip (stretched in x by {x_stretch}x) from r={radius_ch1_inner} to r={radius_ch1_outer} (width={radius_ch1_outer-radius_ch1_inner})")
print(f"  - Channel 2: 3D concentrated cloud points centered at z={z_center} (z_max/2), z spread ±{z_spread}, in ring r=[{radius_ch1_inner}, {radius_ch1_outer}], {num_cloud_points} centers")
print(f"  - Channel 3: 3D concentrated cloud points centered at z={z_center} (z_max/2), z spread ±{z_spread}, near channel 2 centroids (max distance {max_centroid_distance}), {num_cloud_points} centers")
print(f"  - Cloud points: {points_per_cloud} points per cloud, radius = {cloud_radius} (low scattering)")
print(f"\nData type: {data.dtype}")
print(f"Total elements: {data.size:,}")
print(f"Memory size: {data.nbytes / 1024 / 1024:.2f} MB")

# Save the data
np.save('oval_stripes.npy', data)
print(f"\n✓ texture6 (Oval strips and cloud patterns) saved to: oval_stripes.npy")

Created texture6 (Oval strips and cloud patterns): (4, 11, 49, 344, 681)
Dimensions breakdown:
  - Channels: 4 (1, 2, 3, 4)
  - Values: 11 (0 to 10)
  - Z: 49 (0 to 48)
  - Y: 344 (0 to 343)
  - X: 681 (0 to 680)

Texture6 parameters:
  - Center position: (340, 172)
  - Channel 0: Oval strip (stretched in x by 1.5x) from r=50 to r=100 (width=50)
  - Channel 1: Oval strip (stretched in x by 1.5x) from r=80 to r=110 (width=30)
  - Channel 2: 3D concentrated cloud points centered at z=24 (z_max/2), z spread ±10, in ring r=[80, 110], 5 centers
  - Channel 3: 3D concentrated cloud points centered at z=24 (z_max/2), z spread ±10, near channel 2 centroids (max distance 5), 5 centers
  - Cloud points: 100 points per cloud, radius = 20 (low scattering)

Data type: int8
Total elements: 505,073,184
Memory size: 481.68 MB

✓ texture6 (Oval strips and cloud patterns) saved to: oval_stripes.npy


visulaize oval_stipes

In [10]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"
# Load data: shape (channel, value, z, y, x)
data = np.load("oval_stripes.npy")
n_channels, n_values, n_z, y_dim, x_dim = data.shape

# Real ranges
z_range = (0, n_z - 1)  # 0 to 48
x_range = (0, x_dim - 1)  # 0 to 680
y_range = (0, y_dim - 1)  # 0 to 343

# Colors per channel
channel_colors = [
    "#a6cee3",      # channel 0
    "#1f78b4",    # channel 1
    "#b2df8a",     # channel 2
    "#33a02c",  # channel 3
]

# Compute a 3D scalar field per channel: avg over "value"
per_channel_3d = np.zeros((n_channels, n_z, y_dim, x_dim), dtype=np.float32)

for ch in range(n_channels):
    channel_data = data[ch]

    value_indices = np.arange(n_values, dtype=np.float32).reshape(-1, 1, 1, 1)
    weighted_sum = np.sum(value_indices * channel_data, axis=0)
    total_sum = np.sum(channel_data, axis=0)

    avg_value_index = np.zeros_like(weighted_sum)
    mask = total_sum > 0
    avg_value_index[mask] = weighted_sum[mask] / total_sum[mask]

    per_channel_3d[ch] = avg_value_index

# --- Plotly 3D scatter ---
fig = go.Figure()
threshold = 0.1
max_points = 5000

for ch in range(n_channels):
    vol = per_channel_3d[ch]

    if np.all(vol == 0):
        continue

    vmin, vmax = vol.min(), vol.max()
    if vmax <= vmin:
        continue

    norm = (vol - vmin) / (vmax - vmin + 1e-6)
    mask = norm > threshold

    if not np.any(mask):
        continue

    z_idx, y_idx, x_idx = np.where(mask)

    # limit number of points
    n_pts = len(x_idx)
    if n_pts > max_points:
        selected = np.random.choice(n_pts, max_points, replace=False)
        x_idx = x_idx[selected]
        y_idx = y_idx[selected]
        z_idx = z_idx[selected]

    fig.add_trace(go.Scatter3d(
        x=x_idx,
        y=y_idx,
        z=z_idx,
        mode="markers",
        marker=dict(
            size=2,
            color=channel_colors[ch],
            opacity=0.8  # must be single number!
        ),
        name=f"Biomarker {ch}"
    ))

# Calculate aspect ratios to show non-isotropic nature
x_span = x_range[1] - x_range[0] + 1  # 681
y_span = y_range[1] - y_range[0] + 1  # 344
z_span = z_range[1] - z_range[0] + 1  # 49

# Normalize to show relative sizes (non-isotropic)
# Z is much smaller than X and Y
max_span = max(x_span, y_span, z_span)
aspect_x = x_span / max_span  # ~1.0
aspect_y = y_span / max_span  # ~0.51
aspect_z = z_span / max_span  # ~0.072

fig.update_layout(
    title="3D Interactive View of Channels (Non-Isotropic: Z << X, Y)",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        xaxis=dict(range=[x_range[0], x_range[1]]),  # 0 to 680
        yaxis=dict(range=[y_range[0], y_range[1]]),  # 0 to 343
        zaxis=dict(range=[z_range[0], z_range[1]]),  # 0 to 48
        aspectmode="manual",
        aspectratio=dict(x=aspect_x, y=aspect_y, z=aspect_z),
    ),
    autosize=True,
    showlegend=True,
)

print(f"3D Visualization with real ranges (Non-Isotropic):")
print(f"  X range: [{x_range[0]}, {x_range[1]}] (span: {x_span})")
print(f"  Y range: [{y_range[0]}, {y_range[1]}] (span: {y_span})")
print(f"  Z range: [{z_range[0]}, {z_range[1]}] (span: {z_span})")
print(f"  Aspect ratio (X:Y:Z): {aspect_x:.3f}:{aspect_y:.3f}:{aspect_z:.3f}")
print(f"  Note: Z is {x_span/z_span:.1f}x smaller than X and {y_span/z_span:.1f}x smaller than Y")

fig.show(config={
    'displayModeBar': True,
    'displaylogo': False,
    'modeBarButtonsToAdd': ['toImage'],
    'toImageButtonOptions': {
        'format': 'png',
        'filename': '3d_visualization',
        'height': None,
        'width': None,
        'scale': 1
    }
})


3D Visualization with real ranges (Non-Isotropic):
  X range: [0, 680] (span: 681)
  Y range: [0, 343] (span: 344)
  Z range: [0, 48] (span: 49)
  Aspect ratio (X:Y:Z): 1.000:0.505:0.072
  Note: Z is 13.9x smaller than X and 7.0x smaller than Y


create GT from oval stripes

In [2]:
import numpy as np

# ============================================================================
# STEP 1: Load the groundtruth data
# ============================================================================
data = np.load("oval_stripes.npy")
n_channels, n_values, n_z, y_dim, x_dim = data.shape

print(f"Loaded data shape: {data.shape}")
print(f"  Channels: {n_channels} (0-3)")
print(f"  Values: {n_values} (0-10)")
print(f"  Spatial dimensions: Z={n_z}, Y={y_dim}, X={x_dim}")

# ============================================================================
# STEP 2: Find voxel-level intersections (where 2+ channels overlap)
# Intersection = voxel where more than one channel has data
# ============================================================================
print(f"\n{'='*60}")
print(f"STEP 2: Finding voxel-level intersections")
print(f"{'='*60}")

# For each channel, sum over values axis to get presence map
# Input shape: (n_channels, n_values, n_z, y_dim, x_dim)
# Output shape: (n_channels, n_z, y_dim, x_dim)
channel_presence = np.sum(data, axis=1)  # Sum over values

# Create binary presence map (1 if channel has any data at that voxel)
channel_binary = (channel_presence > 0).astype(np.int8)

# Count how many channels are present at each voxel
# Shape: (n_z, y_dim, x_dim)
channels_per_voxel = np.sum(channel_binary, axis=0)

# Intersection voxels: where 2 or more channels are present
intersection_mask = channels_per_voxel >= 2

print(f"Channel presence shape: {channel_binary.shape}")
for ch in range(n_channels):
    print(f"  Channel {ch} non-zero voxels: {np.sum(channel_binary[ch] > 0):,}")
print(f"\nIntersection statistics (voxels with 2+ channels):")
print(f"  Voxels with 2 channels: {np.sum(channels_per_voxel == 2):,}")
print(f"  Voxels with 3 channels: {np.sum(channels_per_voxel == 3):,}")
print(f"  Voxels with 4 channels: {np.sum(channels_per_voxel == 4):,}")
print(f"  Total intersection voxels: {np.sum(intersection_mask):,}")

# ============================================================================
# STEP 3: Get all intersection voxel positions as ground truth
# No cube-based approach - directly use voxel-level intersections
# ============================================================================
print(f"\n{'='*60}")
print(f"STEP 3: Extracting intersection voxel positions")
print(f"{'='*60}")

# Get coordinates of all intersection voxels
intersection_coords = np.where(intersection_mask)
z_coords = intersection_coords[0]
y_coords = intersection_coords[1]
x_coords = intersection_coords[2]

print(f"Total intersection voxels (GT): {len(z_coords):,}")
print(f"  These are voxels where 2 or more channels overlap")

# ============================================================================
# STEP 4: Summary of ground truth selection
# ============================================================================
print(f"\n{'='*60}")
print(f"STEP 4: Ground Truth Summary")
print(f"{'='*60}")

total_voxels = n_z * y_dim * x_dim
intersection_percentage = (len(z_coords) / total_voxels) * 100

print(f"  Total voxels in volume: {total_voxels:,}")
print(f"  Intersection voxels (GT): {len(z_coords):,}")
print(f"  Intersection percentage: {intersection_percentage:.2f}%")
print(f"  (No ranking or scoring - ALL intersections are GT)")

# ============================================================================
# STEP 5: Create new array with 5 channels
# Channels 0-3: Copy from original data
# Channel 4: Mark ALL intersection voxels as GT
# ============================================================================
print(f"\n{'='*60}")
print(f"STEP 5: Creating oval_stripes_GT.npy with 5 channels")
print(f"{'='*60}")

# Create new array with 5 channels (4 original + 1 GT channel)
# Shape: (5, n_values, n_z, y_dim, x_dim)
ground_truth_oval = np.zeros((n_channels + 1, n_values, n_z, y_dim, x_dim), dtype=data.dtype)

# Copy original 4 channels (0-3) from original data
ground_truth_oval[:n_channels] = data

print(f"Created new array with shape: {ground_truth_oval.shape}")
print(f"  Channels 0-3: Copied from original data")
print(f"  Channel 4: GT channel (intersection markers)")

# Add GT values to channel 4 (index 4)
# Mark ALL intersection voxels as GT (where 2+ channels overlap)
gt_channel_idx = n_channels  # Channel 4 (index 4)
gt_value_idx = 0  # Store GT value at value index 0 (marker)

# Mark all intersection voxels in GT channel using vectorized operation
ground_truth_oval[gt_channel_idx, gt_value_idx, z_coords, y_coords, x_coords] = 1

print(f"  Marked {len(z_coords):,} intersection voxels as GT")

# ============================================================================
# STEP 6: Save the ground truth file
# ============================================================================
print(f"\n{'='*60}")
print(f"STEP 6: Saving oval_stripes_GT.npy")
print(f"{'='*60}")

np.save('oval_stripes_GT.npy', ground_truth_oval)

print(f"✓ Ground truth with 5 channels saved to: oval_stripes_GT.npy")
print(f"  Final shape: {ground_truth_oval.shape}")
print(f"  Channels: 0-3 (original), 4 (GT channel)")
print(f"  Number of intersection voxels (GT): {len(z_coords):,}")
print(f"  GT voxels in channel 4: {np.sum(ground_truth_oval[gt_channel_idx] > 0):,}")
print(f"  Intersection = voxels where 2 or more channels overlap")

print(f"\n{'='*60}")
print(f"Process completed successfully!")
print(f"{'='*60}")

Loaded data shape: (4, 11, 49, 344, 681)
  Channels: 4 (0-3)
  Values: 11 (0-10)
  Spatial dimensions: Z=49, Y=344, X=681

STEP 2: Finding voxel-level intersections
Channel presence shape: (4, 49, 344, 681)
  Channel 0 non-zero voxels: 1,732,542
  Channel 1 non-zero voxels: 1,316,630
  Channel 2 non-zero voxels: 500
  Channel 3 non-zero voxels: 500

Intersection statistics (voxels with 2+ channels):
  Voxels with 2 channels: 832,118
  Voxels with 3 channels: 549
  Voxels with 4 channels: 0
  Total intersection voxels: 832,667

STEP 3: Extracting intersection voxel positions
Total intersection voxels (GT): 832,667
  These are voxels where 2 or more channels overlap

STEP 4: Ground Truth Summary
  Total voxels in volume: 11,478,936
  Intersection voxels (GT): 832,667
  Intersection percentage: 7.25%
  (No ranking or scoring - ALL intersections are GT)

STEP 5: Creating oval_stripes_GT.npy with 5 channels
Created new array with shape: (5, 11, 49, 344, 681)
  Channels 0-3: Copied from orig

visulize oval strips_GT

In [5]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"
# Load data: shape (channel, value, z, y, x)
data = np.load("oval_stripes_GT.npy")
n_channels, n_values, n_z, y_dim, x_dim = data.shape

print(f"Loaded oval_stripes_GT.npy shape: {data.shape}")
print(f"  Channels: {n_channels} (0-3: original, 4: GT spot)")

# Real ranges
z_range = (0, n_z - 1)  # 0 to 48
x_range = (0, x_dim - 1)  # 0 to 680
y_range = (0, y_dim - 1)  # 0 to 343

# Colors per channel
channel_colors = [
    "#a6cee3",      # channel 0 - light blue
    "#1f78b4",      # channel 1 - blue
    "#b2df8a",      # channel 2 - light green
    "#33a02c",      # channel 3 - green
    "#000000",      # channel 4 - black (GT channel)
]

# Compute a 3D scalar field per channel: avg over "value"
per_channel_3d = np.zeros((n_channels, n_z, y_dim, x_dim), dtype=np.float32)

# Process channels 0-3 (original channels)
for ch in range(n_channels - 1):  # Process channels 0-3
    channel_data = data[ch]

    value_indices = np.arange(n_values, dtype=np.float32).reshape(-1, 1, 1, 1)
    weighted_sum = np.sum(value_indices * channel_data, axis=0)
    total_sum = np.sum(channel_data, axis=0)

    avg_value_index = np.zeros_like(weighted_sum)
    mask = total_sum > 0
    avg_value_index[mask] = weighted_sum[mask] / total_sum[mask]

    per_channel_3d[ch] = avg_value_index

# Process channel 4 (GT spot)
gt_channel = data[n_channels - 1]  # Channel 4
# GT spots are stored at value index 0
gt_spots = gt_channel[0]  # Shape: (n_z, y_dim, x_dim)
per_channel_3d[n_channels - 1] = gt_spots.astype(np.float32)

# --- Plotly 3D scatter ---
fig = go.Figure()
threshold = 0.5
max_points = 5000

# Plot channels 0-3 (original channels)
for ch in range(n_channels - 1):
    vol = per_channel_3d[ch]

    if np.all(vol == 0):
        continue

    vmin, vmax = vol.min(), vol.max()
    if vmax <= vmin:
        continue

    norm = (vol - vmin) / (vmax - vmin + 1e-6)
    mask = norm > threshold

    if not np.any(mask):
        continue

    z_idx, y_idx, x_idx = np.where(mask)

    # limit number of points
    n_pts = len(x_idx)
    if n_pts > max_points:
        selected = np.random.choice(n_pts, max_points, replace=False)
        x_idx = x_idx[selected]
        y_idx = y_idx[selected]
        z_idx = z_idx[selected]

    fig.add_trace(go.Scatter3d(
        x=x_idx,
        y=y_idx,
        z=z_idx,
        mode="markers",
        marker=dict(
            size=2,
            color=channel_colors[ch],
            opacity=0.8
        ),
        name=f"Channel {ch}"
    ))

# Plot channel 4 (GT channel) - black spots
gt_channel_idx = n_channels - 1
vol = per_channel_3d[gt_channel_idx]

if np.any(vol > 0):
    z_idx, y_idx, x_idx = np.where(vol > 0)
    total_gt_points = len(z_idx)
    
    # Limit GT points same as other channels (same max_points ratio)
    n_pts = len(x_idx)
    if n_pts > max_points:
        selected = np.random.choice(n_pts, max_points, replace=False)
        x_idx = x_idx[selected]
        y_idx = y_idx[selected]
        z_idx = z_idx[selected]
    
    fig.add_trace(go.Scatter3d(
        x=x_idx,
        y=y_idx,
        z=z_idx,
        mode="markers",
        marker=dict(
            size=2,  # Same size as other channels
            color=channel_colors[gt_channel_idx],  # Black
            opacity=0.4,
            line=dict(width=0)
        ),
        name="GT spot"
    ))
    print(f"  GT spot: {len(z_idx)} displayed (total: {total_gt_points}, max_points: {max_points})")

# Calculate aspect ratios to show non-isotropic nature
x_span = x_range[1] - x_range[0] + 1  # 681
y_span = y_range[1] - y_range[0] + 1  # 344
z_span = z_range[1] - z_range[0] + 1  # 49

# Normalize to show relative sizes (non-isotropic)
# Z is much smaller than X and Y
max_span = max(x_span, y_span, z_span)
aspect_x = x_span / max_span  # ~1.0
aspect_y = y_span / max_span  # ~0.51
aspect_z = z_span / max_span  # ~0.072

fig.update_layout(
    title="3D Visualization: All Channels (0-3) + GT spot",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        xaxis=dict(range=[x_range[0], x_range[1]]),  # 0 to 680
        yaxis=dict(range=[y_range[0], y_range[1]]),  # 0 to 343
        zaxis=dict(range=[z_range[0], z_range[1]]),  # 0 to 48
        aspectmode="manual",
        aspectratio=dict(x=aspect_x, y=aspect_y, z=aspect_z),
        bgcolor="white"
    ),
    autosize=True,
    showlegend=True,
)

print(f"\n3D Visualization Summary:")
print(f"  Channels 0-3: Original channels with colored markers")
print(f"  GT spot: Black spots (size=5)")
print(f"  X range: [{x_range[0]}, {x_range[1]}] (span: {x_span})")
print(f"  Y range: [{y_range[0]}, {y_range[1]}] (span: {y_span})")
print(f"  Z range: [{z_range[0]}, {z_range[1]}] (span: {z_span})")
print(f"  Aspect ratio (X:Y:Z): {aspect_x:.3f}:{aspect_y:.3f}:{aspect_z:.3f}")
print(f"  Note: Z is {x_span/z_span:.1f}x smaller than X and {y_span/z_span:.1f}x smaller than Y")

fig.show(config={
    'displayModeBar': True,
    'displaylogo': False,
    'modeBarButtonsToAdd': ['toImage'],
    'toImageButtonOptions': {
        'format': 'png',
        'filename': '3d_visualization',
        'height': None,
        'width': None,
        'scale': 1
    }
})


Loaded oval_stripes_GT.npy shape: (5, 11, 49, 344, 681)
  Channels: 5 (0-3: original, 4: GT spot)
  GT spot: 5000 displayed (total: 832667, max_points: 5000)

3D Visualization Summary:
  Channels 0-3: Original channels with colored markers
  GT spot: Black spots (size=5)
  X range: [0, 680] (span: 681)
  Y range: [0, 343] (span: 344)
  Z range: [0, 48] (span: 49)
  Aspect ratio (X:Y:Z): 1.000:0.505:0.072
  Note: Z is 13.9x smaller than X and 7.0x smaller than Y
